# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
%pip -q install duckdb huggingface_hub pandas numpy scikit-learn

In [2]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

In [3]:
HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

Paste your Hugging Face READ token (hf_...): ··········


In [4]:
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    "dim_clients":                f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content":                f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d":             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
    "fact_daily_sample":          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
}

In [5]:
for name, table in TABLES.items():

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    schema = con.sql(
        f"DESCRIBE SELECT * FROM {table}"
    ).df()

    display(schema)


dim_clients


,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,is_active,BOOLEAN,YES,None,None,None
2,has_gsc_access,BOOLEAN,YES,None,None,None
3,has_ga4_access,BOOLEAN,YES,None,None,None
4,access_profile,VARCHAR,YES,None,None,None
5,client_created_date,DATE,YES,None,None,None
6,client_updated_date,DATE,YES,None,None,None
7,gsc_data_start,DATE,YES,None,None,None
8,ga4_data_start,DATE,YES,None,None,None



dim_content


,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None



fact_daily


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None



fact_query_90d


,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,query_hash_id,VARCHAR,YES,None,None,None
3,query_char_count,BIGINT,YES,None,None,None
4,query_token_count,BIGINT,YES,None,None,None
5,window_start,DATE,YES,None,None,None
6,window_end,DATE,YES,None,None,None
7,impressions_90d,BIGINT,YES,None,None,None
8,clicks_90d,BIGINT,YES,None,None,None
9,impressions_last30,BIGINT,YES,None,None,None



fact_daily_sample


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [6]:
column_inventory = []

for table_name, table in TABLES.items():

    schema = con.sql(
        f"DESCRIBE SELECT * FROM {table}"
    ).df()

    for _, row in schema.iterrows():

        column_inventory.append({
            "table": table_name,
            "column": row["column_name"],
            "dtype": row["column_type"]
        })

column_inventory = pd.DataFrame(column_inventory)

print(
    f"Total table-column combinations: "
    f"{len(column_inventory):,}"
)

display(column_inventory)

Total table-column combinations: 118


,table,column,dtype
0,dim_clients,client_hash_id,VARCHAR
1,dim_clients,is_active,BOOLEAN
2,dim_clients,has_gsc_access,BOOLEAN
3,dim_clients,has_ga4_access,BOOLEAN
4,dim_clients,access_profile,VARCHAR
...,...,...,...
113,fact_daily_sample,ai_claude,BIGINT
114,fact_daily_sample,ai_meta,BIGINT
115,fact_daily_sample,ai_other,BIGINT
116,fact_daily_sample,scroll_events,BIGINT


In [7]:
table_counts = []

for name, table in TABLES.items():

    n = con.sql(
        f"SELECT COUNT(*) AS n FROM {table}"
    ).fetchone()[0]

    table_counts.append({
        "table": name,
        "rows": n
    })

table_counts = pd.DataFrame(table_counts)

display(table_counts)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,table,rows
0,dim_clients,104
1,dim_content,519606
2,fact_daily,78835655
3,fact_query_90d,2414248
4,fact_daily_sample,11694072


In [8]:
for table_name in ["fact_daily", "fact_query_90d", "fact_daily_sample"]:

    print("\n" + "=" * 80)
    print(table_name)
    print("=" * 80)

    if table_name == "fact_daily":
        date_column = "report_date"

    elif table_name == "fact_daily_sample":
        date_column = "report_date"

    elif table_name == "fact_query_90d":
        date_column = None

    if date_column:

        result = con.sql(
            f"""
            SELECT
                MIN({date_column}) AS min_date,
                MAX({date_column}) AS max_date,
                COUNT(*) AS rows
            FROM {TABLES[table_name]}
            """
        ).df()

        display(result)

    else:

        result = con.sql(
            f"""
            SELECT
                MIN(window_start) AS min_window_start,
                MAX(window_start) AS max_window_start,
                MIN(window_end) AS min_window_end,
                MAX(window_end) AS max_window_end,
                COUNT(*) AS rows
            FROM {TABLES[table_name]}
            """
        ).df()

        display(result)


fact_daily


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min_date,max_date,rows
0,2025-01-27,2026-06-30,78835655



fact_query_90d


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min_window_start,max_window_start,min_window_end,max_window_end,rows
0,2026-04-02,2026-04-02,2026-06-30,2026-06-30,2414248



fact_daily_sample


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min_date,max_date,rows
0,2026-06-01,2026-06-30,11694072


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [9]:
FEATURE_START = "2026-03-01"
FEATURE_END   = "2026-04-01"

TARGET_START  = "2026-04-01"
TARGET_END    = "2026-05-01"

PREDICTION_DATE = TARGET_START

In [10]:
features = con.sql(
    f"""
    WITH feature_window AS (

        SELECT
            client_hash_id,
            content_hash_id,

            SUM(gsc_impressions) AS imp_prev30,

            SUM(gsc_clicks) AS clk_prev30,

            AVG(gsc_avg_position) AS avg_position_prev30,

            COUNT(*) FILTER (
                WHERE gsc_impressions > 0
            ) AS days_with_impressions

        FROM {TABLES["fact_daily"]}

        WHERE report_date >= DATE '{FEATURE_START}'
          AND report_date < DATE '{FEATURE_END}'

          AND gsc_data_available IS TRUE

        GROUP BY
            client_hash_id,
            content_hash_id

    )

    SELECT

        f.client_hash_id,
        f.content_hash_id,

        f.imp_prev30,
        f.clk_prev30,
        f.avg_position_prev30,
        f.days_with_impressions,

        DATE_DIFF(
            'day',
            c.content_created_date,
            DATE '{PREDICTION_DATE}'
        ) AS content_age_days

    FROM feature_window f

    LEFT JOIN {TABLES["dim_content"]} c
        ON f.client_hash_id = c.client_hash_id
       AND f.content_hash_id = c.content_hash_id

    WHERE f.imp_prev30 >= 100
    """
).df()

print(f"Feature rows: {len(features):,}")

display(features.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature rows: 101,441


,client_hash_id,content_hash_id,imp_prev30,clk_prev30,avg_position_prev30,days_with_impressions,content_age_days
0,client_2094c6eb080311d5,content_14c17f59aa610ab3,122.0,2.0,5.882498,8,43
1,client_2094c6eb080311d5,content_15565677b6e1792f,148.0,1.0,12.350803,16,21
2,client_2094c6eb080311d5,content_15770c63daac443b,2585.0,4.0,6.251382,23,56
3,client_2094c6eb080311d5,content_157db5a38382c639,1143.0,4.0,6.104022,20,56
4,client_2094c6eb080311d5,content_15f758d8bd5341c5,347.0,0.0,46.934948,31,43


In [11]:
features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101441 entries, 0 to 101440
Data columns (total 7 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   client_hash_id         101441 non-null  object 
 1   content_hash_id        101441 non-null  object 
 2   imp_prev30             101441 non-null  float64
 3   clk_prev30             101441 non-null  float64
 4   avg_position_prev30    101441 non-null  float64
 5   days_with_impressions  101441 non-null  int64  
 6   content_age_days       101441 non-null  int64  
dtypes: float64(3), int64(2), object(2)
memory usage: 5.4+ MB


In [12]:
features.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
client_hash_id,101441,44,client_73cda7b4e4f265ea,21633,NaN,NaN,NaN,NaN,NaN,NaN,NaN
content_hash_id,101441,101441,content_3858d289ed7069ea,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
imp_prev30,101441.0,NaN,NaN,NaN,2748.357006,6945.163794,100.0,283.0,786.0,2514.0,617124.0
clk_prev30,101441.0,NaN,NaN,NaN,8.037924,34.887274,0.0,0.0,1.0,6.0,5668.0
avg_position_prev30,101441.0,NaN,NaN,NaN,14.438976,14.284799,0.013793,4.990232,8.699482,19.177628,93.693397
days_with_impressions,101441.0,NaN,NaN,NaN,28.50032,5.022045,1.0,29.0,31.0,31.0,31.0
content_age_days,101441.0,NaN,NaN,NaN,189.937146,127.258728,2.0,72.0,188.0,266.0,495.0


In [13]:
feature_missing = (
    features
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .to_frame("missing_pct")
)

display(feature_missing)

,missing_pct
client_hash_id,0.0
content_hash_id,0.0
imp_prev30,0.0
clk_prev30,0.0
avg_position_prev30,0.0
days_with_impressions,0.0
content_age_days,0.0


In [14]:
for col in [
    "imp_prev30",
    "clk_prev30",
    "avg_position_prev30",
    "days_with_impressions",
    "content_age_days"
]:

    print("\n", "=" * 70)
    print(col)
    print("=" * 70)

    print(
        features[col]
        .isna()
        .value_counts(dropna=False)
    )


imp_prev30
imp_prev30
False    101441
Name: count, dtype: int64

clk_prev30
clk_prev30
False    101441
Name: count, dtype: int64

avg_position_prev30
avg_position_prev30
False    101441
Name: count, dtype: int64

days_with_impressions
days_with_impressions
False    101441
Name: count, dtype: int64

content_age_days
content_age_days
False    101441
Name: count, dtype: int64


### Feature notes

| Feature                 | Meaning                                                                  | Missing-value handling                                                                                    | Available when?                                                                                |
| ----------------------- | ------------------------------------------------------------------------ | --------------------------------------------------------------------------------------------------------- | ---------------------------------------------------------------------------------------------- |
| `imp_prev30`            | Total GSC impressions during the March feature window                    | Pages are restricted to at least 100 impressions; no missing values after the GSC availability filter     | Before the prediction moment because all observations are from the previous 30 days            |
| `clk_prev30`            | Total GSC clicks during the March feature window                         | Retained as the observed aggregate; zero clicks are treated as zero observed clicks when GSC is available | Before the prediction moment because the clicks belong to the completed historical window      |
| `avg_position_prev30`   | Average GSC search position during the March feature window              | Missing values are median-imputed for modeling when no valid position measurement exists                  | Before the prediction moment because the position observations come from the historical window |
| `days_with_impressions` | Number of March days on which the page recorded positive GSC impressions | No imputation required for pages retained in the feature frame                                            | Before the prediction moment because it is calculated entirely from the historical window      |
| `content_age_days`      | Number of days between content creation and the prediction moment        | Missing content dates are median-imputed for the first model                                              | Before the prediction moment because the content creation date is historical metadata          |


In [15]:
future_outcomes = con.sql(
    f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS imp_next30

    FROM {TABLES["fact_daily"]}

    WHERE report_date >= DATE '{TARGET_START}'
      AND report_date < DATE '{TARGET_END}'

      AND gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
    """
).df()

display(future_outcomes.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,imp_next30
0,client_62f4a7e64f5e0096,content_76c1f31e2b38f054,249.0
1,client_62f4a7e64f5e0096,content_ffc5ab4b34aab1f8,281.0
2,client_62f4a7e64f5e0096,content_9739856fc83dc1ca,722.0
3,client_62f4a7e64f5e0096,content_3d1dc691a3502105,3068.0
4,client_62f4a7e64f5e0096,content_9d28af4f99c5e67b,3896.0


In [16]:
data = features.merge(
    future_outcomes,
    on=[
        "client_hash_id",
        "content_hash_id"
    ],
    how="inner"
)

print(f"Rows with both feature and target windows: {len(data):,}")

Rows with both feature and target windows: 100,893


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [17]:
data["impression_change_pct"] = (
    (data["imp_next30"] - data["imp_prev30"])
    / data["imp_prev30"]
)

In [18]:
data["is_declining"] = (
    data["impression_change_pct"] < -0.20
).astype(int)

In [19]:
display(
    data["is_declining"]
    .value_counts()
    .rename({
        0: "not_declining",
        1: "declining"
    })
)

,count
is_declining,
declining,51944
not_declining,48949


In [22]:
all_columns = column_inventory["column"].drop_duplicates().tolist()

suspicious_keywords = [
    "future",
    "next",
    "trend",
    "change",
    "declin",
    "priority",
    "health",
    "action",
    "refresh",
    "flag",
    "target",
    "label"
]

suspicious_columns = [
    col
    for col in all_columns
    if any(keyword in col.lower() for keyword in suspicious_keywords)
]

print("Potentially suspicious columns:")
for col in sorted(suspicious_columns):
    print("-", col)

Potentially suspicious columns:


In [23]:
product_keywords = [
    "health",
    "priority",
    "action",
    "refresh",
    "flag"
]

product_columns = [
    col
    for col in all_columns
    if any(k in col.lower() for k in product_keywords)
]

print("Product / decision-like fields:")
for col in sorted(product_columns):
    print("-", col)

Product / decision-like fields:


The warehouse release does not provide FlyRank's product decision fields such as ``health_score, priority_score, or action_type,`` so these cannot be used as features.

In [25]:
query_overlap = con.sql(
    f"""
    SELECT
        COUNT(*) AS overlapping_rows
    FROM {TABLES["fact_query_90d"]}
    WHERE window_end >= DATE '{FEATURE_END}'
    """
).df()

display(query_overlap)

,overlapping_rows
0,2414248


In [26]:
query_future_rows = con.sql(
    f"""
    SELECT
        COUNT(*) AS rows_using_future_query_data
    FROM {TABLES["fact_query_90d"]}
    WHERE window_end >= DATE '{FEATURE_END}'
    """
).df()

display(query_future_rows)

,rows_using_future_query_data
0,2414248


In [27]:
future_feature_rows = con.sql(
    f"""
    SELECT COUNT(*) AS suspicious_rows
    FROM {TABLES["fact_daily"]}
    WHERE report_date >= DATE '{TARGET_START}'
      AND report_date < DATE '{TARGET_END}'
      AND gsc_data_available IS TRUE
    """
).df()

display(future_feature_rows)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,suspicious_rows
0,3901060


The query-level table contains fixed 90-day windows. Many windows extend beyond the March 31 prediction point. Therefore, query-level aggregates from this table cannot automatically be treated as March features. They are excluded from the first feature vector until a historical query snapshot whose ``window_end`` is at or before the prediction moment is available.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [28]:
column_decisions = [
    # dim_clients
    ("dim_clients", "client_hash_id", "context",
     "Identifier used for joins and client-level validation."),

    ("dim_clients", "is_active", "excluded",
     "Client status is not required for the first Search Intelligence vector."),

    ("dim_clients", "has_gsc_access", "context",
     "Data availability context, not a page-performance feature."),

    ("dim_clients", "has_ga4_access", "context",
     "Data availability context, not required for this GSC-based lane."),

    ("dim_clients", "access_profile", "excluded",
     "Client-level metadata is excluded from the first vector."),

    ("dim_clients", "client_created_date", "context",
     "Client lifecycle information."),

    ("dim_clients", "client_updated_date", "excluded",
     "Not required for the first vector and may reflect later administrative updates."),

    ("dim_clients", "gsc_data_start", "context",
     "Used to understand historical coverage."),

    ("dim_clients", "ga4_data_start", "context",
     "Used to understand analytics coverage."),

    # dim_content
    ("dim_content", "client_hash_id", "context",
     "Join key."),

    ("dim_content", "content_hash_id", "context",
     "Page identifier."),

    ("dim_content", "keyword_hash_id", "context",
     "Scrambled identifier; grouping only."),

    ("dim_content", "url_hash_id", "context",
     "Scrambled identifier; grouping only."),

    ("dim_content", "content_created_date", "feature-source",
     "Used to calculate content_age_days at the prediction moment."),

    ("dim_content", "content_updated_date", "excluded",
     "Current metadata may reflect information after the prediction moment."),

    ("dim_content", "content_type", "excluded",
     "Not required for the first vector."),

    ("dim_content", "search_volume", "excluded",
     "Not required for the first vector and timing of the measurement is not established here."),

    ("dim_content", "competition", "excluded",
     "Not required for the first vector."),

    ("dim_content", "competition_level", "excluded",
     "Not required for the first vector."),

    ("dim_content", "cpc", "excluded",
     "Not required for the first vector."),

    ("dim_content", "main_intent", "excluded",
     "Categorical metadata excluded from the first vector."),

    ("dim_content", "backlinks", "excluded",
     "Current value may not represent the state at the prediction moment."),

    ("dim_content", "category_count", "excluded",
     "Not required for the first vector."),

    ("dim_content", "keyword_created_date", "excluded",
     "Not required for the first vector."),

    ("dim_content", "provider_used", "excluded",
     "Product/content-generation metadata is not required for this lane."),

    ("dim_content", "model_used", "excluded",
     "Product/content-generation metadata is not required for this lane."),

    ("dim_content", "char_count", "excluded",
     "Current metadata may not represent the historical prediction-time state."),

    ("dim_content", "word_count", "excluded",
     "Current metadata may not represent the historical prediction-time state."),

    ("dim_content", "last_optimized_date", "excluded",
     "Current optimization metadata may contain post-decision information."),

    ("dim_content", "optimization_eligible_date", "excluded",
     "Not required for the first vector."),

    ("dim_content", "is_published", "excluded",
     "Current status may not represent the prediction-time state."),

    ("dim_content", "is_deleted", "excluded",
     "Current status may contain post-prediction information."),

    # daily features
    ("fact_daily", "gsc_impressions", "feature",
     "Aggregated over the completed previous 30-day feature window."),

    ("fact_daily", "gsc_clicks", "feature",
     "Aggregated over the completed previous 30-day feature window."),

    ("fact_daily", "gsc_avg_position", "feature",
     "Aggregated over the completed previous 30-day feature window."),

    ("fact_daily", "ga4_pageviews", "excluded",
     "Not required for the first GSC-based vector."),

    ("fact_daily", "ga4_sessions", "excluded",
     "Not required for the first GSC-based vector."),

    ("fact_daily", "ga4_users", "excluded",
     "Not required for the first GSC-based vector."),

    ("fact_daily", "ga4_engaged_sessions", "excluded",
     "Not required for the first GSC-based vector."),

    ("fact_daily", "ga4_total_engagement_sec", "excluded",
     "Not required for the first GSC-based vector."),

    ("fact_daily", "sessions_organic", "excluded",
     "Not required for the first GSC-based vector."),

    ("fact_daily", "sessions_direct", "excluded",
     "Not required for the first GSC-based vector."),

    ("fact_daily", "sessions_referral", "excluded",
     "Not required for the first GSC-based vector."),

    ("fact_daily", "sessions_social", "excluded",
     "Not required for the first GSC-based vector."),

    ("fact_daily", "sessions_paid", "excluded",
     "Not required for the first GSC-based vector."),

    ("fact_daily", "sessions_ai", "excluded",
     "Not required for the first GSC-based vector."),

    ("fact_daily", "scroll_events", "excluded",
     "Not required for the first GSC-based vector."),

    ("fact_daily", "gsc_data_available", "context",
     "Used to ensure that missing GSC tracking is not treated as zero performance."),

    ("fact_daily", "ga4_data_available", "context",
     "Availability context only."),

    ("fact_daily", "client_has_gsc", "context",
     "Availability context only."),

    ("fact_daily", "client_has_ga4", "context",
     "Availability context only."),

    ("fact_daily", "report_date", "context",
     "Defines the feature and target windows."),

    # query table
    ("fact_query_90d", "window_start", "excluded",
     "Fixed query windows may overlap the future target period."),

    ("fact_query_90d", "window_end", "excluded",
     "Used to detect temporal overlap; not a model feature."),

    ("fact_query_90d", "impressions_90d", "excluded",
     "Fixed 90-day window is not aligned to the March prediction moment."),

    ("fact_query_90d", "clicks_90d", "excluded",
     "Fixed 90-day window is not aligned to the March prediction moment."),

    ("fact_query_90d", "impressions_last30", "excluded",
     "The table's fixed window is not guaranteed to be historical relative to March."),

    ("fact_query_90d", "clicks_last30", "excluded",
     "The table's fixed window is not guaranteed to be historical relative to March."),

    ("fact_query_90d", "avg_position_90d", "excluded",
     "Fixed 90-day window may overlap the target period."),

    ("fact_query_90d", "avg_position_last30", "excluded",
     "Fixed window timing does not match the March decision point."),

    ("fact_query_90d", "rare_impressions_share", "excluded",
     "Potentially useful signal, but its fixed window is not safely aligned to the prediction moment."),

    ("fact_query_90d", "anonymized_impressions_share", "excluded",
     "Potentially useful signal, but its fixed window is not safely aligned to the prediction moment."),

    ("fact_query_90d", "content_visible_query_count", "excluded",
     "Potentially useful signal, but its fixed window is not safely aligned to the prediction moment."),

    # label
    ("constructed", "imp_next30", "label-source",
     "Future outcome used to construct the decline label."),

    ("constructed", "impression_change_pct", "label-derived",
     "Uses future impressions and therefore cannot be a feature."),

    ("constructed", "is_declining", "label",
     "Target: future impressions decline by more than 20%.")
]

column_decisions_df = pd.DataFrame(
    column_decisions,
    columns=[
        "table",
        "column",
        "decision",
        "reason"
    ]
)

display(column_decisions_df)

,table,column,decision,reason
0,dim_clients,client_hash_id,context,Identifier used for joins and client-level val...
1,dim_clients,is_active,excluded,Client status is not required for the first Se...
2,dim_clients,has_gsc_access,context,"Data availability context, not a page-performa..."
3,dim_clients,has_ga4_access,context,"Data availability context, not required for th..."
4,dim_clients,access_profile,excluded,Client-level metadata is excluded from the fir...
...,...,...,...,...
61,fact_query_90d,anonymized_impressions_share,excluded,"Potentially useful signal, but its fixed windo..."
62,fact_query_90d,content_visible_query_count,excluded,"Potentially useful signal, but its fixed windo..."
63,constructed,imp_next30,label-source,Future outcome used to construct the decline l...
64,constructed,impression_change_pct,label-derived,Uses future impressions and therefore cannot b...


### What I excluded and why

I deliberately excluded several classes of warehouse fields from the first Search Intelligence feature vector.

* **Client and content hash IDs** — identifiers used for joins and grouping, not meaningful measurements.
* **`report_date`** — used to define the historical and future windows, not a predictive feature by itself.
* **GSC/GA4 availability flags** — used to distinguish missing tracking from zero activity; they are context rather than page-performance features.
* **Future-window impressions and clicks** — these are observed only after the prediction moment and are used only to construct the label.
* **`impression_change_pct`** — deliberately demonstrated as a leakage feature because it uses the future target window directly.
* **`trend_direction` / `trend_pct` when based on future or overlapping observations** — they can contain information from the outcome period.
* **`fact_content_query_90d` features** — excluded from the first vector because the table contains fixed 90-day windows whose `window_end` is not guaranteed to be at or before the March prediction moment. Using the supplied June window would directly expose future information.
* **Product decision fields such as priority, health, action, refresh, or similar flags** — excluded because they represent product decisions rather than independent observable signals; the warehouse release does not provide these fields.
* **Current content metadata whose historical value cannot be established** — fields such as current optimization status, current word count, or current deletion status may reflect changes after the prediction moment, so they are excluded from this first honest vector.
* **GA4 and AI-referral measurements** — excluded from this first Search Intelligence vector because the target is GSC-based search decline and the first iteration is intentionally limited to five features.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.